# Double Integrals and Monte Carlo

A double integral can be computed as an iterated integral, by adaptive cubature, or by random sampling. Cubature wins in low dimensions and Monte Carlo in high ones.

In [ ]:
using HCubature, QuadGK, Statistics, Random, Plots
Random.seed!(1)
f(x) = exp(-(x[1]^2 + x[2]^2))
lo, hi = [0.0, 0.0], [1.0, 1.0]
exact = (sqrt(pi) / 2 * 0.8427007929497149)^2   # (integral of exp(-t^2) over [0, 1]) squared

## Iterated integrals (Fubini)

In [ ]:
inner(u) = quadgk(v -> f([u, v]), lo[2], hi[2])[1]
(iterated = quadgk(inner, lo[1], hi[1])[1], exact = exact)

## Adaptive cubature

In [ ]:
hcubature_count(f, lo, hi; rtol=1e-10)

## Monte Carlo

Average $f$ at random points and multiply by the volume. The standard error is $|\Omega|\,s_f/\sqrt{N}$.

In [ ]:
function montecarlo(f, lo, hi, N)
    vol = prod(hi .- lo)
    ys = [f(lo .+ (hi .- lo) .* rand(length(lo))) for _ in 1:N]
    return vol * mean(ys), vol * std(ys) / sqrt(N)
end
[(N = N, estimate = montecarlo(f, lo, hi, N)[1], standard_error = montecarlo(f, lo, hi, N)[2], error = montecarlo(f, lo, hi, N)[1] - exact) for N in (10^2, 10^4, 10^6)]

## The curse of dimensionality

Integrate $\exp(-\lVert x \rVert^2)$ over $[0,1]^d$: the exact value is the one-dimensional integral to the power $d$. Watch how the evaluations needed by cubature explode with $d$, while Monte Carlo's error at fixed $N$ barely moves.

In [ ]:
one = sqrt(pi) / 2 * 0.8427007929497149
[(d = d, cubature_evaluations = hcubature_count(x -> exp(-sum(abs2, x)), zeros(d), ones(d); rtol=1e-6)[3],
  monte_carlo_error_1e5 = abs(montecarlo(x -> exp(-sum(abs2, x)), zeros(d), ones(d), 10^5)[1] - one^d)) for d in 1:6]

## Exercises

1. Integrate over the unit disc by multiplying $f$ by an indicator of $x_1^2 + x_2^2 \le 1$. What happens to cubature's accuracy?
2. Compute the expected utility $E[\log(1 + X_1 + X_2)]$ for independent uniform shocks on $[0, 1]$.